In [1]:
from typing import Literal, Dict, List, TypedDict

import shared_libraries.data_processing as processing

import pandas as pd

In [2]:



CompoundType = Literal["SOFT", "MEDIUM", "HARD"]
RealCompoundType = Literal["C1", "C2", "C3", "C4", "C5"]

class CompoundMapping(TypedDict):
    SOFT: RealCompoundType
    MEDIUM: RealCompoundType
    HARD: RealCompoundType

WindDirectionType = Literal["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

class Weather(TypedDict):
    AirTemp: float
    Humidity: int
    Pressure: float
    Rainfall: bool
    TrackTemp: float
    WindSpeed: float
    WindDirection: WindDirectionType


class Lap(TypedDict):
    LapNumber: int
    TyreLife: int
    Compound: CompoundType
    IsPitLap: bool

class Strategy:
    def __init__(self, initial_compound: CompoundType, stops: Dict[int, CompoundType]) -> None:
        self.initial_compound: CompoundType = initial_compound
        self.stops = stops

def prepare_strategies() -> List[Strategy]:
    return [
        Strategy("SOFT", {20: "MEDIUM"}),
        Strategy("MEDIUM", {25: "HARD"})
    ]

def create_strategy_dataframe(strategies: List[Strategy]) -> pd.DataFrame:
    return pd.DataFrame({
        "Id": list(range(len(strategies))),
        "Strategy": strategies
    })

def prepare_lap_data(strategy_dataframe: pd.DataFrame, race_length: int) -> pd.DataFrame:
    race_dfs = []
    for idx in strategy_dataframe["Id"].array:
        race_laps = _get_race(strategy_dataframe.loc[idx, "Strategy"], race_length) # type: ignore
        race_df = pd.concat(
            [
                pd.DataFrame({
                    "StrategyId": [idx] * len(race_laps)
                }),
                pd.DataFrame(race_laps)
            ]
        , axis="columns")
        race_dfs.append(race_df)
    return pd.concat(race_dfs, axis="index", ignore_index=True)
    

def get_real_compounds(lap_data: pd.DataFrame, compound_mapping: CompoundMapping) -> pd.Series:
    return lap_data["Compound"].map(lambda compound: compound_mapping[compound])

def get_weather_dataframe(lap_data: pd.DataFrame, weather: Weather) -> pd.DataFrame:
    return pd.DataFrame([weather] * lap_data.shape[0])




def _get_race(strategy: Strategy, race_length: int) -> List[Lap]:
    laps: List[Lap] = []
    compound = strategy.initial_compound
    tyre_life = 1
    for lap_number in range(1, race_length + 1):
        if lap_number in strategy.stops:
            is_pit_lap = True
        else:
            is_pit_lap = False

        laps.append({
            "LapNumber": lap_number,
            "TyreLife": tyre_life,
            "Compound": compound,
            "IsPitLap": is_pit_lap
        })

        if is_pit_lap:
            compound = strategy.stops[lap_number]
            tyre_life = 1
        else:
            tyre_life += 1


    return laps











In [3]:
import pickle
with open("./ig_data/data_by_circuit.pickle", "rb") as file:
    data_by_circuit = pickle.load(file)

In [4]:
weather_data: pd.DataFrame = data_by_circuit["Sakhir"].copy().loc[
    :, 
    [
        "AirTemp",
        "Humidity",
        "Pressure",
        "Rainfall",
        "TrackTemp",
        "WindSpeed",
    ]
]

weather = weather_data.mean().to_dict()
weather["WindDirection"] = "N"


In [5]:
strategies = prepare_strategies()
strategies

[<__main__.Strategy at 0x158ef016cd0>, <__main__.Strategy at 0x158f10c8510>]

In [6]:
strategies_df = create_strategy_dataframe(strategies)
strategies_df

,Id,Strategy
0,0,<__main__.Strategy object at 0x00000158EF016CD0>
1,1,<__main__.Strategy object at 0x00000158F10C8510>


In [7]:
lap_data = prepare_lap_data(strategies_df, 60)
lap_data

,StrategyId,LapNumber,TyreLife,Compound,IsPitLap
0,0,1,1,SOFT,False
1,0,2,2,SOFT,False
2,0,3,3,SOFT,False
3,0,4,4,SOFT,False
4,0,5,5,SOFT,False
...,...,...,...,...,...
115,1,56,31,HARD,False
116,1,57,32,HARD,False
117,1,58,33,HARD,False
118,1,59,34,HARD,False


In [8]:
real_compounds = get_real_compounds(lap_data, compound_mapping={
    "SOFT": "C1",
    "MEDIUM": "C3",
    "HARD": "C5"
})
real_compounds

0      C1
1      C1
2      C1
3      C1
4      C1
       ..
115    C5
116    C5
117    C5
118    C5
119    C5
Name: Compound, Length: 120, dtype: object

In [9]:
weather_df = get_weather_dataframe(lap_data, weather) # type: ignore
weather_df

,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindSpeed,WindDirection
0,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N
1,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N
2,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N
3,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N
4,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N
...,...,...,...,...,...,...,...
115,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N
116,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N
117,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N
118,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N


In [10]:
full_lap_data = pd.concat([lap_data, weather_df], axis="columns")
full_lap_data["RealCompound"] = real_compounds

full_lap_data


,StrategyId,LapNumber,TyreLife,Compound,IsPitLap,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindSpeed,WindDirection,RealCompound
0,0,1,1,SOFT,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C1
1,0,2,2,SOFT,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C1
2,0,3,3,SOFT,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C1
3,0,4,4,SOFT,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C1
4,0,5,5,SOFT,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1,56,31,HARD,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C5
116,1,57,32,HARD,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C5
117,1,58,33,HARD,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C5
118,1,59,34,HARD,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,N,C5


In [13]:
dummies = pd.get_dummies(full_lap_data)
dummies

,StrategyId,LapNumber,TyreLife,IsPitLap,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindSpeed,Compound_HARD,Compound_MEDIUM,Compound_SOFT,WindDirection_N,RealCompound_C1,RealCompound_C3,RealCompound_C5
0,0,1,1,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,False,False,True,True,True,False,False
1,0,2,2,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,False,False,True,True,True,False,False
2,0,3,3,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,False,False,True,True,True,False,False
3,0,4,4,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,False,False,True,True,True,False,False
4,0,5,5,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,False,False,True,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1,56,31,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,True,False,False,True,False,False,True
116,1,57,32,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,True,False,False,True,False,False,True
117,1,58,33,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,True,False,False,True,False,False,True
118,1,59,34,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,True,False,False,True,False,False,True


In [14]:
processing.add_missing_dummy_columns(dummies)
dummies

,StrategyId,LapNumber,TyreLife,IsPitLap,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindSpeed,...,HARD,RealCompound_C2,RealCompound_C4,WindDirection_NE,WindDirection_E,WindDirection_SE,WindDirection_S,WindDirection_SW,WindDirection_W,WindDirection_NW
0,0,1,1,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
1,0,2,2,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
2,0,3,3,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
3,0,4,4,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
4,0,5,5,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1,56,31,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
116,1,57,32,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
117,1,58,33,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
118,1,59,34,False,23.575953,38.273101,1013.290052,0.0,27.76475,0.794136,...,False,False,False,False,False,False,False,False,False,False
